# 03 – Build and Test the Model

Goal: build a model that guesses which machines will fail in the next 7 days. The model only uses what was known on the day of each reading. It learns from older months and is tested on newer months.

### 1. Load the clean data

In [41]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/machine_observations_clean.csv",
                 parse_dates=["observation_timestamp"])
print(df.shape)

(10334, 15)


### 2. Split by time
Train on older readings and test on the latest readings, like real use. A 7-day gap between them stops the target of the last training rows from seeing into the test period.
**Look for:** enough failures in the test set to judge the model fairly.

In [42]:
test_start = pd.Timestamp("2024-11-01")
train_end = test_start - pd.Timedelta(days=7)   # 7-day gap

train = df[df["observation_timestamp"] < train_end]
test = df[df["observation_timestamp"] >= test_start]

for name, part in [("Train", train), ("Test", test)]:
    print(name)
    print("  Dates:   ", part["observation_timestamp"].min().date(), "to", part["observation_timestamp"].max().date())
    print("  Rows:    ", len(part))
    print("  Failures:", part["fails_within_7d"].sum())
    print("  Rate %:  ", round(part["fails_within_7d"].mean() * 100, 1))
    print()

print("Share of rows in test:", round(len(test) / (len(train) + len(test)) * 100, 1), "%")
y_test = test["fails_within_7d"]

Train
  Dates:    2024-01-01 to 2024-10-24
  Rows:     7238
  Failures: 349
  Rate %:   4.8

Test
  Dates:    2024-11-01 to 2025-02-28
  Rows:     2922
  Failures: 151
  Rate %:   5.2

Share of rows in test: 28.8 %


### Why not accuracy?
Only about 5% of test readings have a failure coming. A lazy model that always says "no failure" visits no machines and finds no failures, but its accuracy is still about 95%.
So this notebook measures what matters to the team:
- **Recall:** of all broken machines, how many are found?
- **Precision:** of all visits, how many find a broken machine?

In [43]:
lazy_accuracy = (y_test == 0).mean()
print("Accuracy of a model that always says 'no failure':", round(lazy_accuracy * 100, 1), "%")
print("Broken machines it finds: 0 of", y_test.sum())

Accuracy of a model that always says 'no failure': 94.8 %
Broken machines it finds: 0 of 151


### 3. Which sensor works best alone? (training data)
Before building a model, a simple rule is needed to compare it with. This is called the **baseline**. If the model cannot beat a simple rule, it is not worth using.

The simplest rule is to use one sensor as the clue: sort the readings by that sensor, and visit from the top of the list. This cell finds which sensor is the best clue on its own.

**How it works:** for each sensor, sort the training readings (highest first, and also lowest first), then go down the list until 70% of the real failures are found. Count the visits needed. **Fewer visits = better clue.**

Only the trainning data is used here, so the test data stays unseen.

In [44]:
import math

def go_down_the_list(y_true, score, share_to_find):
    """Sort readings by score (highest first), go down the list
    until the chosen share of real failures is found, and count."""
    y_sorted = np.asarray(y_true)[np.argsort(-np.asarray(score), kind="stable")]
    total_failures = int(y_sorted.sum())
    target = math.ceil(share_to_find * total_failures)
    found_so_far = np.cumsum(y_sorted)
    visits = int(np.argmax(found_so_far >= target)) + 1
    found = int(found_so_far[visits - 1])
    return {"visits": visits,
            "found": found,
            "healthy visited": visits - found,
            "missed": total_failures - found,
            "useful %": round(found / visits * 100, 1)}

y_train = train["fails_within_7d"]
y_test = test["fails_within_7d"]

sensors = ["load_pct", "ambient_temp_c", "vibration_mm_s",
           "bearing_temp_c", "motor_current_a", "discharge_pressure_kpa"]

rows = []
for col in sensors:
    values = train[col].fillna(train[col].median())
    high_first = go_down_the_list(y_train, values, 0.7)    # high value = risky
    low_first = go_down_the_list(y_train, -values, 0.7)    # low value = risky
    best = high_first if high_first["visits"] <= low_first["visits"] else low_first
    rows.append({"sensor": col,
                 "risky when": "high" if best is high_first else "low",
                 **best})

pd.DataFrame(rows).sort_values("visits")

,sensor,risky when,visits,found,healthy visited,missed,useful %
5,discharge_pressure_kpa,low,1717,245,1472,104,14.3
2,vibration_mm_s,high,1726,245,1481,104,14.2
3,bearing_temp_c,high,3600,245,3355,104,6.8
4,motor_current_a,high,4428,245,4183,104,5.5
1,ambient_temp_c,high,4724,245,4479,104,5.2
0,load_pct,low,5028,245,4783,104,4.9


### 4. Baseline on the test data
Cell 3 showed that pressure and vibration are the best single clues, and almost tied. So both are used as simple rules (the baseline):
- **Lowest pressure first:** visit the machines with the lowest pressure first.
- **Highest vibration first:** visit the machines that shake the most first.

Each rule is scored on the test data: go down the list until 106 of the 151 broken machines are found (70%), and count the visits. The model in Cell 5 must need **fewer** visits than these rules to be useful.

In [45]:
results = {}

pressure = test["discharge_pressure_kpa"].fillna(train["discharge_pressure_kpa"].median())
results["Lowest pressure first"] = go_down_the_list(y_test, -pressure, 0.7)

vib = test["vibration_mm_s"].fillna(train["vibration_mm_s"].median())
results["Highest vibration first"] = go_down_the_list(y_test, vib, 0.7)

pd.DataFrame(results).T

,visits,found,healthy visited,missed,useful %
Lowest pressure first,699.0,106.0,593.0,45.0,15.2
Highest vibration first,754.0,106.0,648.0,45.0,14.1


### 5. Main model: logistic regression
The model uses all the safe clues together: the sensor readings, `unplanned_stops_14d`, site and machine class. It does NOT use `corrective_orders_14d` or `inspection_interval_days` (see Notebook 02).
It learns from the training data how many points each clue is worth, then gives every test reading a risk score. The test readings are sorted by that score.
Missing values are filled with the training median. `class_weight="balanced"` makes the model pay extra attention to the rare failures.

In [46]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

numeric = ["load_pct", "ambient_temp_c", "vibration_mm_s", "bearing_temp_c",
           "motor_current_a", "discharge_pressure_kpa", "unplanned_stops_14d"]
category = ["site", "machine_class"]

def build_model(numeric_cols, model):
    prepare = make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric_cols),
        (OneHotEncoder(handle_unknown="ignore"), category),
    )
    return make_pipeline(prepare, model)

lr = build_model(numeric, LogisticRegression(max_iter=1000, class_weight="balanced"))
lr.fit(train[numeric + category], y_train)
lr_score = lr.predict_proba(test[numeric + category])[:, 1]

results["Logistic regression (main model)"] = go_down_the_list(y_test, lr_score, 0.7)
pd.DataFrame(results).T

,visits,found,healthy visited,missed,useful %
Lowest pressure first,699.0,106.0,593.0,45.0,15.2
Highest vibration first,754.0,106.0,648.0,45.0,14.1
Logistic regression (main model),376.0,106.0,270.0,45.0,28.2


### 6. Compare with a more complex model: random forest
Same clues as the main model. A random forest is 300 yes/no decision trees that vote.
**Look for:** if it is not clearly better (fewer visits), keep the simpler logistic regression, which is easier to explain.

In [47]:
from sklearn.ensemble import RandomForestClassifier

rf = build_model(numeric, RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                                 class_weight="balanced", random_state=42))
rf.fit(train[numeric + category], y_train)
rf_score = rf.predict_proba(test[numeric + category])[:, 1]

results["Random forest"] = go_down_the_list(y_test, rf_score, 0.7)
pd.DataFrame(results).T

,visits,found,healthy visited,missed,useful %
Lowest pressure first,699.0,106.0,593.0,45.0,15.2
Highest vibration first,754.0,106.0,648.0,45.0,14.1
Logistic regression (main model),376.0,106.0,270.0,45.0,28.2
Random forest,401.0,106.0,295.0,45.0,26.4


### 7. What if the untrusted columns are added?
Same logistic regression, plus `corrective_orders_14d` and `inspection_interval_days`.
**Look for:** a big drop in visits. That would show how much the model depends on columns that may be recorded because of the breakdown (see Notebook 02).

In [48]:
risky_numeric = numeric + ["corrective_orders_14d", "inspection_interval_days"]

lr_risky = build_model(risky_numeric, LogisticRegression(max_iter=1000, class_weight="balanced"))
lr_risky.fit(train[risky_numeric + category], y_train)
risky_score = lr_risky.predict_proba(test[risky_numeric + category])[:, 1]

results["With untrusted columns (not trusted)"] = go_down_the_list(y_test, risky_score, 0.7)
pd.DataFrame(results).T

,visits,found,healthy visited,missed,useful %
Lowest pressure first,699.0,106.0,593.0,45.0,15.2
Highest vibration first,754.0,106.0,648.0,45.0,14.1
Logistic regression (main model),376.0,106.0,270.0,45.0,28.2
Random forest,401.0,106.0,295.0,45.0,26.4
With untrusted columns (not trusted),141.0,106.0,35.0,45.0,75.2


### 8. How far down the list should the team go?
Using the main model's list. Going further down finds more failures, but visits more healthy machines.
All numbers are totals over the 17 test weeks.

In [49]:
weeks = (test["observation_timestamp"].max() - test["observation_timestamp"].min()).days / 7

rows = []
for share in [0.5, 0.6, 0.7, 0.8, 0.9]:
    r = go_down_the_list(y_test, lr_score, share)
    r["goal"] = f"find {int(share * 100)}% of broken machines ({r['found']} of 151)"
    r["visits per week"] = round(r["visits"] / weeks, 1)
    rows.append(r)

pd.DataFrame(rows).set_index("goal")

,visits,found,healthy visited,missed,useful %,visits per week
goal,,,,,,
find 50% of broken machines (76 of 151),195,76,119,75,39.0,11.5
find 60% of broken machines (91 of 151),283,91,192,60,32.2,16.6
find 70% of broken machines (106 of 151),376,106,270,45,28.2,22.1
find 80% of broken machines (121 of 151),524,121,403,30,23.1,30.8
find 90% of broken machines (136 of 151),812,136,676,15,16.7,47.8


### 9. Results and recommendation

**Why 70%?**
All methods are compared at the same goal: find 70% of the broken machines (106 of 151).
- The brief says missing a failure is costly, so the goal should be to find **most** broken machines, not just a few.
- Finding almost all of them is very expensive. With the model, finding 90% needs 812 visits, more than double the 376 visits needed for 70% (see Cell 8).
- 70% is a middle point between those two concerns.

The exact level is a business decision. Cell 8 shows the results at 50% to 90%, so the team can choose.

**Visits needed to find 106 of the 151 broken machines (test data, 17 weeks)**

| How machines are picked | Visits | Wasted visits (machine was fine) |
|---|---|---|
| Lowest pressure first | 699 | 593 |
| Highest vibration first | 756 | 650 |
| **Logistic regression (main model)** | **376** | **270** |
| Random forest | 401 | 295 |
| With untrusted columns | 141 (not trusted) | 35 |

**Findings**
- The main model needs about half the visits of the best simple rule to find the same broken machines.
- Random forest was not better. The patterns are fairly simple (for example, lower pressure means more risk), and there are only about 350 failures to learn from, so a more complex model adds little. The simpler and more explainable logistic regression is kept.
- With the untrusted columns, the model needs only 141 visits. This is not trusted, because these columns may be recorded because of the breakdown (see Notebook 02).

**Recommendation: not ready for deployment, but worth further development.**

Reasons:
- **Too many wasted visits.** Even with the model, 270 of the 376 visits (about 7 in 10) find a machine that is fine. The team may stop trusting the alerts.
- **Two columns cannot be used yet.** It is not known whether repair orders and inspection intervals are in the system at the moment of the reading, or written after a breakdown. Only the data owner can confirm this.
- **Data errors were found.** South pressure changed unit, some vibration values were −999, and some rows were copied. New data could have the same errors, so automatic data checks are needed before real use.

**Next steps**
1. **Ask the data owner** when repair orders, stops and failures are entered into the system, and which date is stored. This decides whether those columns can be used.
2. **If the columns are confirmed safe,** test the model again with them, since they could greatly reduce wasted visits.
3. **Run a small trial at one site,** comparing the model's alerts with what the team really finds, to check it works in real life.
